In [2]:
!pip install findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Broadcast Variables Parte 4").getOrCreate()
sc = spark.sparkContext

# Variables de Difusión (*Broadcast Variables*)

En Apache Spark, cuando ejecutamos una operación en un clúster, Spark envía automáticamente una copia de las variables utilizadas en la función a cada uno de los **tasks** (tareas). Si tienes miles de tareas ejecutándose en tus nodos, y esas tareas usan un conjunto de datos grande (como un diccionario de búsqueda o una tabla de parámetros), copiar esos datos una y otra vez a través de la red destruirá el rendimiento de tu aplicación.

Para resolver esto existen las **Broadcast Variables**.

---

## ¿Qué son y cómo funcionan?

Las *Broadcast Variables* te permiten mantener una **variable de solo lectura guardada en caché en cada máquina (nodo)**, en lugar de enviar una copia de ella con cada tarea. Spark se asegura de que los datos se envíen a través de la red a cada nodo **una sola vez**, utilizando algoritmos eficientes para reducir el costo de comunicación.


### La Analogía del Salón de Clases 🏫
* **Sin Broadcast Variables:** El profesor (Driver) le entrega una copia impresa del diccionario completo a cada alumno (Task) cada vez que hacen una pregunta. ¡Mucho papel y tiempo desperdiciado!
* **Con Broadcast Variables:** El profesor coloca un diccionario en la mesa de cada salón de clases (Nodo/Ejecutor). Todos los alumnos de ese salón comparten el mismo diccionario sin necesidad de que cada uno tenga el suyo.

---

## Ventajas Clave

* 🚀 **Ahorro de Red:** Los datos viajan una sola vez por nodo, no por tarea

In [3]:
rdd = sc.parallelize([item for item in range(10)])
rdd.collect()

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [4]:
uno = 1

In [5]:
# va a poner a disposicion de todos los ejecutores esta informacion
br_broadcast1 = sc.broadcast(uno)

In [6]:
# hemos accedido a su valor ahora se le sumo el elemento 10 a la variabale broadcast

rdd1 = rdd.map(lambda x: x + br_broadcast1.value)
rdd1.collect()

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Liberación de Recursos: El método `unpersist()` en Variables Broadcast

Aunque las *Broadcast Variables* son altamente eficientes, debemos recordar que **ocupan espacio en la memoria RAM de todos los ejecutores**. Si el volumen de datos difundido es muy grande, esto podría llegar a causar problemas de saturación de recursos (errores de *Out of Memory*) en el clúster.

Para mitigar esto, Spark nos permite liberar esa memoria de forma manual cuando ya no necesitemos los datos de manera inmediata.

---

## El método `unpersist()`

Para limpiar la memoria de los ejecutores, simplemente debemos llamar al método `unpersist()` sobre la variable broadcast que hayamos creado previamente:

```python
mi_variable_broadcast.unpersist()

In [7]:
br_broadcast1.unpersist()

In [9]:
rdd1 = rdd.map(lambda x: x + br_broadcast1.value)
rdd1.collect()

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [10]:
# se puede eliminar por completo en todo el cluster?

br_broadcast1.destroy()

In [11]:
rdd1 = rdd.map(lambda x: x + br_broadcast1.value)


In [ ]:
# se elimino por completo la variable boradcast

rdd1.take(4)